# Build Synthetic Dataset
Pipeline: researchers → projects (GPT-4o) → assignments → preprocessing → validation

In [ ]:
# On Kaggle: uncomment the line below
# !pip install hazm openai python-dotenv -q

import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / '.env')

from src import config
from src.researchers import generate_researchers
from src.projects import generate_projects, get_openai_client
from src.assignments import assign_roles, compute_specialty_weights
from src.preprocess import save_preprocessed

print('Imports OK')
print('DATA_RAW:', config.DATA_RAW)

In [ ]:
print('Generating 100 researchers...')
researchers_df = generate_researchers()
print(f'Done: {len(researchers_df)} researchers')
print(researchers_df[['researcher_id', 'academic_rank', 'self_declared_specialties']].head())

In [ ]:
print('Generating 1300 projects via GPT-4o (may take 30-60 min)...')
client = get_openai_client()
projects_df = generate_projects(researchers_df, client=client)
print(f'Done: {len(projects_df)} projects')
print(projects_df['difficulty'].value_counts())

In [ ]:
print('Assigning roles (manager / supervisor / collaborator)...')
assignments_df = assign_roles(projects_df, researchers_df)
print(f'Done: {len(assignments_df)} assignment records')
print(assignments_df['role'].value_counts())

In [ ]:
print('Computing specialty weights from project history...')
researchers_df = compute_specialty_weights(projects_df, assignments_df, researchers_df)
print('Sample weights:', researchers_df['specialty_weights'].iloc[0])

In [ ]:
print('Preprocessing text with Hazm...')
clean_df = save_preprocessed(projects_df)
print(f'Done: {len(clean_df)} clean rows')
print('Sample clean_text:', clean_df['clean_text'].iloc[0][:150])

In [ ]:
import subprocess
result = subprocess.run(
    ['pytest', 'tests/test_dataset.py', '-v', '--tb=short'],
    capture_output=True, text=True,
    cwd=str(repo_root)
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])